# FinSent Phase 3 — encoder grid on Colab

3 models (ModernBERT-base / DeBERTa-v3-base / FinBERT) × seeds {1, 2, 3} × train sets {50agree, allagree, 50agree-n1569} = **27 runs**.

- **Runtime → Change runtime type → GPU**（T4 够用；L4 / A100 更快，且支持 bf16）
- `outputs/preds` 和 `outputs/logs` 链接到 Google Drive，**断线后从头重跑所有 cell 即可续跑**：`run_grid` 会跳过 preds 已存在的 run
- checkpoint 只在 Colab 本地磁盘，每个 run 结束后自动删除；val curve 存在 `logs/<run>.trainer_state.json`
- ⚠️ 不要 `pip install` 整个项目依赖：`torch>=2.15` 会覆盖 Colab 自带的 CUDA 版 torch

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive（存结果，防断线）

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Clone / update repo

In [ ]:
import os
REPO = "/content/finsent"
if not os.path.exists(REPO):
    !git clone -q https://github.com/FrankJia2007/finsent.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1

## 4. Install（不重装 torch）

In [ ]:
!pip install -e . --no-deps -q
!pip install -q "transformers>=5.0" datasets accelerate sentencepiece

## 5. 把 outputs/preds 和 outputs/logs 链接到 Drive

In [ ]:
DRIVE_OUT = "/content/drive/MyDrive/finsent_outputs"
os.makedirs(f"{REPO}/outputs", exist_ok=True)
for sub in ["preds", "logs"]:
    target, link = f"{DRIVE_OUT}/{sub}", f"{REPO}/outputs/{sub}"
    os.makedirs(target, exist_ok=True)
    if os.path.islink(link):
        continue
    if os.path.exists(link):
        raise RuntimeError(f"{link} 已是普通目录；先确认里面没有要保留的结果，再手动删掉")
    os.symlink(target, link)
!ls -la {REPO}/outputs
print("already on Drive:", len(os.listdir(f"{DRIVE_OUT}/preds")), "preds files")

## 6. Sanity check：版本、GPU、精度、路径

In [ ]:
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0),
      "| bf16 supported:", torch.cuda.is_bf16_supported())
from finsent.data import PROCESSED_DIR, OUTPUTS_DIR
print(PROCESSED_DIR, "→", sorted(p.name for p in PROCESSED_DIR.glob("*.parquet")))
print(OUTPUTS_DIR)

## 7. Run the grid

先 dry run 看计划，再正式跑。每个 run 一行：`[i/N] name ... ok  x.x min`

In [ ]:
!python -m finsent.run_grid --with_control --dry_run

In [ ]:
!python -m finsent.run_grid --with_control

## 8. Quick sanity check

不是正式 benchmark（Phase 5 用 `summarize()` + bootstrap CI 汇总），只是检查有没有坏掉的 run：
- 只预测 < 3 个类别 → 很可能训练崩了（参考 DeBERTa fp16 NaN 事件）
- val loss 出现 NaN

In [ ]:
import glob, json, math
import pandas as pd
from finsent.evaluate import macro_f1

rows = []
for f in sorted(glob.glob(f"{REPO}/outputs/preds/*.parquet")):
    d = pd.read_parquet(f)
    clean = d[d["all_agree"]]
    name = os.path.basename(f)[:-len(".parquet")]
    state_f = f"{REPO}/outputs/logs/{name}.trainer_state.json"
    val = []
    if os.path.exists(state_f):
        val = [h for h in json.load(open(state_f))["log_history"] if "eval_macro_f1" in h]
    rows.append({"run": name, "model": d["model"].iloc[0], "seed": d["seed"].iloc[0],
                 "test=allagree": macro_f1(clean["label"], clean["pred"]),
                 "test=50agree": macro_f1(d["label"], d["pred"]),
                 "n_pred_classes": d["pred"].nunique(),
                 "best_epoch": max(val, key=lambda h: h["eval_macro_f1"])["epoch"] if val else None,
                 "val_nan": any(math.isnan(h["eval_loss"]) for h in val)})
res = pd.DataFrame(rows)
display(res.groupby("model")[["test=allagree", "test=50agree"]].agg(["mean", "std", "count"]).round(3))
bad = res[(res["n_pred_classes"] < 3) | res["val_nan"]]
print("suspicious runs:" if len(bad) else "no suspicious runs ✓")
if len(bad):
    display(bad)

## 9. 下载结果

本地解压到 `finsent/outputs/`（会得到 `outputs/preds/` 和 `outputs/logs/`）。Drive 上的 `finsent_outputs/` 也是一份完整备份。

In [ ]:
!cd {REPO}/outputs && zip -qr /content/finsent_outputs.zip preds logs && ls -lh /content/finsent_outputs.zip
from google.colab import files
files.download("/content/finsent_outputs.zip")